In [ ]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.ops as ops

from torch import Tensor

In [ ]:
# Squeeze-and-Excitation attention
ops.SqueezeExcitation(2, 1)

In [ ]:
# MultiHead Attention
# L   -> target sequence length
# S   -> source sequence length
# E_q -> is the query embedding dimension (embed_dim)
# E_k -> key embeddings dim (kdim)
# E_v -> value embeddings dim (vdim
embed_dim = 256
num_heads = 8
MHA = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
query = torch.Tensor()  # Query embeddings  (N, L, E_q)
key =   torch.Tensor()  # Key embeddings    (N, S, E_k)
value = torch.Tensor()  # Value embeddings  (N, S, E_v)
attn_output, attn_output_weights = MHA(query, key, value)
# NOTE: In self-attention, query, key, and value tensors are the same

In [ ]:
# Embedding
sentence = 'The quick brown fox jumps over a lazy dog'
dc = {s: i for i, s in enumerate(sorted(sentence.replace(',', '').split()))}
# print(dc)

r = [dc[i] for i in sentence.replace(',', '').split()]
sentence_int = torch.tensor(r)

vocab_size = 50000  # Assume a large vocabulary size
torch.manual_seed(123)
embed = nn.Embedding(vocab_size, 3)
embedded_sentence = embed(sentence_int).detach()
print(embedded_sentence)
print(embedded_sentence.shape)
embed(torch.tensor([1])).detach()

In [ ]:
# Self Attention
class SelfAttention(nn.Module):
    def __init__(self, d, d_q, d_k, d_v):
        super(SelfAttention, self).__init__()
        self.d = d
        self.d_q = d_q
        self.d_k = d_k
        self.d_v = d_v
        self.W_query = nn.Parameter(torch.rand(d, d_q))
        self.W_key =   nn.Parameter(torch.rand(d, d_k))
        self.W_value = nn.Parameter(torch.rand(d, d_v))
    
    def forward(self, x):
        Q = x @ self.W_query
        K = x @ self.W_key
        V = x @ self.W_value
        attention_scores = Q @ K.T / np.sqrt(self.d_k)
        attention_weights = F.softmax(attention_scores, dim=-1)
        context_vector = attention_weights @ V
        return context_vector

In [ ]:
# ResNet forward with MHA
from torchvision import models
from torchvision.models import ResNet

# create ResNet with modified classifier
num_classes = 3
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)


# MHA augmented model classifier
class FC_MHA(nn.Module):
    def __init__(self, fc: nn.Module, embed_dim: int, num_heads: int):
        super().__init__()
        self.fc = fc
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.flatten(x, 2)
        x = torch.transpose(x, 1, 2)
        x, _ = self.mha(x, x, x)
        x = x.mean(dim=1)
        x = self.fc(x)
        return x

E = 256
H = 8
model.fc = FC_MHA(model.fc, E, H) # type: ignore

In [ ]:
# MHA modifications

def MHA_forward_ResNet(self, x: Tensor) -> Tensor:

    # Backbone
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.maxpool(x)

    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)

    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    
    # x = torch.flatten(x, 2)
    # x = torch.transpose(x, 1, 2)
    # x, _ = self.mha(x, x, x)
    # x = x.mean(dim=1)
    
    x = self.fc(x)
    return x


E = 256 # channel dimensions of features fed into MHA
H = 8
model.mha = nn.MultiheadAttention(embed_dim=E, num_heads=H, batch_first=True)
model._forward_impl = MHA_forward_ResNet

funcType = type(ResNet._forward_impl)
# model._forward_impl = funcType(MHA_forward_ResNet, model, ResNet)


In [ ]:
# Example of MHA augmented forward (ChatGPT)
def forward(self, x):
    feats = self.backbone(x)        # (N, C, H, W)
    
    feats = feats.flatten(2)        # (N, C, H*W)
    feats = feats.transpose(1, 2)   # (N, L, E) where L=H*W, E=C

    attn_out, _ = self.mha(feats, feats, feats)  # self-attention

    feats = attn_out.mean(dim=1)   # global pooling over sequence
    out = self.fc(feats)
    return out

In [ ]:
# ResNet modified with MHA
from torch import Tensor
from torchvision.models.resnet import ResNet, BasicBlock

class ResNet_MHA(ResNet):
    # def __init__(self):
    #     super().__init__()
    
    def _forward_impl(self, x: Tensor) -> Tensor:

        # backbone
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # attention
        x = torch.flatten(x, 2)
        x = torch.transpose(x, 1, 2)
        x, _ = self.mha(x, x, x)
        x = x.mean(dim=1)

        # classifier
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

def _resnet_MHA():
    return ResNet_MHA(BasicBlock, [2, 2, 2, 2])


#### EffecientNet MHA

In [ ]:
# default loading
model = models.efficientnet_b0(weights=None)
layer_fc: nn.Linear = model.classifier[1] # type: ignore[assignment]
model.classifier[1] = nn.Linear(layer_fc.in_features, num_classes)

In [ ]:
from typing import Any
from torchvision.models import EfficientNet

class EfficientNet_MHA(EfficientNet):
    def __init__(
            self,
            num_heads: int=8,
            **kwargs: Any,
        ):
            super().__init__(**kwargs)
            embed_dim = 1280 # NOTE: According to ChatGPT, must verify!
            self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
    
    def attention(self, x: Tensor) -> Tensor:
        x = x.flatten(2).transpose(1, 2) # (N, H*W, C)
        x, _ = self.mha(x, x, x)
        x = torch.mean(x, dim=1) # (N, 1, C)
        return x
    
    def _forward_impl(self, x: Tensor) -> Tensor:
        x = self.features(x)    # (N, C, W, H)

        # x = self.avgpool(x)     # (N, C, 1, 1)
        x = self.attention(x)
        x = torch.flatten(x, 1) # (N, C)

        x = self.classifier(x)
        return x

In [ ]:
from torchvision.models.efficientnet import _efficientnet_conf

# get instance of efficientnet_b0_MHA
def efficientnet_b0_MHA(
    **kwargs: Any
) -> EfficientNet_MHA:

    inverted_residual_setting, last_channel = _efficientnet_conf("efficientnet_b0", width_mult=1.0, depth_mult=1.0)

    return EfficientNet_MHA(
        inverted_residual_setting=inverted_residual_setting,
        dropout=0.2,
        last_channel=last_channel,
        **kwargs,
    )